# Patterns of Recorded Civilian Targeting in Gaza and the West Bank
## Phase 1: Dataset Selection and Business Problem
**DS312 - Data Mining and Application | Group 7 | A.Y. 2026-2027**  
**Revised:** October 24, 2026

Our study examines the recurrence, geographic concentration, methods, and recorded actor involvement in events targeting Palestinian civilians. We selected ACLED event records from January 1, 2022 to September 30, 2025 to identify places and periods that warrant further human rights investigation.


### 1.1 Business Problem Canvas

| Item | Our project |
|---|---|
| Context / organization | We frame our study around the needs of human rights documentation teams monitoring violence affecting Palestinian civilians. |
| Problem statement | Human rights researchers need reproducible evidence about where recorded violence targeting Palestinian civilians recurs and when it becomes unusually intense or persistent. Our study will identify geographic concentrations, recorded actor involvement, and unusual periods to support further documentary investigation. |
| Stakeholders | Human rights researchers, civilian-protection analysts, investigative journalists, and research coordinators. |
| Decision supported | Our findings will help researchers prioritize locations, periods, and incident groups for closer source verification and contextual investigation. |
| Success criteria | We will assess the usefulness of our results through interpretable geographic clusters, a documented list of unusual district-weeks, comparisons with simple baselines, and stability across settings and precision restrictions. Our investigation recommendations will be traceable to supporting records, and we will report patterns that do not remain consistent. |
| Task type | Data mining through spatial clustering and temporal anomaly detection. |
| Risks and limitations | Our analysis is limited by underreporting, changing source coverage, approximate event locations, uncertain dates, structural missingness, and reported fatality estimates. The records do not capture every instance of civilian harm, and statistical patterns alone cannot establish intent or criminal responsibility. |

We will address the following business questions:

**BQ1.** How do recorded Palestinian civilian-targeting counts and their share of recorded political violence change before and after October 7, 2023, separately in Gaza and the West Bank?

**BQ2.** Which recorded actors and event/sub-event types recur in the focal events, and which actor roles can be supported by the coding and incident notes?

**BQ3.** Where do focal events cluster geographically, and do those concentrations remain evident when location precision and time periods are varied?

**BQ4.** Which district-weeks show unusually high activity or sustained increases relative to earlier local observations?

**BQ5.** How sensitive are the patterns to cohort definitions, baselines, geographic aggregation, and reporting or precision limitations?

We focus primarily on Gaza and analyse the West Bank separately because the two areas have different conditions. Our broader motivation is to examine evidence relevant to systematic attacks on Palestinians. We define a systematic pattern as measurable recurrence, persistence, concentration, or spread in recorded activity. Our study examines these patterns rather than assigning a numerical genocide score. Specific intent and legal classification require a separate documentary and legal assessment.


### 1.2 Dataset Description

| Attribute | Our dataset |
|---|---|
| Dataset / producer | ACLED (Armed Conflict Location & Event Data), event-level records. |
| Source | [Official Data Export Tool](https://acleddata.com/conflict-data/data-export-tool?grain=disaggregated). |
| Download date | October 18, 2026. |
| Requested coverage | Palestine; January 1, 2022-September 30, 2025; all event types; no actor or civilian-targeting filter; one actor per row disabled; default fatality range 0-4,000. |
| Size | 58,842 records and 31 columns; 58,815 records fall within the land-based Gaza/West Bank scope. |
| Unit of observation | One coded event record. Multiple reported actions can be combined under ACLED's aggregation rules, so a row is not necessarily one attack, weapon release, or victim. |
| Collection method | Trained researchers code reported events using media, institutional reports, local partners, and selected verified sources. |
| Data types | Event identifiers, categorical actor/event/location fields, narrative text, numeric fatality estimates, date strings, coordinates, and precision codes. |
| Local raw file | `data/raw/acled_palestine_events_2022_2025.csv`. |
| Reproducibility | We preserved the dated original and recorded the filters and SHA-256 fingerprint in `data/metadata/acled_events_download_manifest.json`. Historical records may be revised by ACLED. |
| Licence | Subject to ACLED's EULA, Content Usage Terms, and Attribution Policy. The dataset is not licensed CC BY 4.0. Raw-data redistribution requires checking applicable permissions. |

We selected this period to include approximately 21 months before October 7, 2023 and two years afterward. We ended the period at a complete month within the available data. Our findings will describe this historical snapshot rather than current conditions.


### 1.3 Initial Data Load and Missing-Value Check

We loaded the original dataset and examined its dimensions, data types, missing values, and duplicate records. We calculated missing-value counts and percentages to support our evaluation of dataset complexity. We preserved the raw CSV throughout this inspection.


In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/raw/acled_palestine_events_2022_2025.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from the project folder containing data/raw/.')
DATA_PATH = ROOT / 'data/raw/acled_palestine_events_2022_2025.csv'
raw = pd.read_csv(DATA_PATH, low_memory=False)
raw_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
manifest = json.loads((ROOT / 'data/metadata/acled_events_download_manifest.json').read_text(encoding='utf-8'))
assert raw_hash == manifest['sha256'], 'CSV differs from the documented download.'
assert raw.shape == (manifest['rows'], len(manifest['columns'])), 'Download dimensions differ from manifest.'
print('Source: ACLED | Downloaded October 18, 2026')
display(raw.head())


Source: ACLED | Downloaded October 18, 2026


,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,actor2,assoc_actor_2,inter2,interaction,civilian_targeting,iso,region,country,admin1,admin2,admin3,location,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp
0,PSE13133,2022-01-01,2022,1,Strategic developments,Strategic developments,Other,Military Forces of Israel (2021-2022),NaN,External/Other forces,Civilians (Palestine),Fishers (Palestine),Civilians,External/Other forces-Civilians,NaN,275,Middle East,Palestine,Gaza Strip,North Gaza,NaN,Coast of North Gaza,31.5839,34.4659,2,Palestine News and Information Agency; PCHR: P...,Other-National,"Other: On 1 January 2022, Israeli gunboats ope...",0,Foreign military engagement,1753954689
1,PSE13135,2022-01-01,2022,1,Demonstrations,Protests,Peaceful protest,Protesters (Palestine),Commission of Detainees and Ex-Detainees Affairs,Protesters,NaN,NaN,NaN,Protesters only,NaN,275,Middle East,Palestine,West Bank,Hebron,NaN,Hebron,31.5294,35.0938,1,Palestine News and Information Agency,National,"On 1 January 2022, Palestinian protesters held...",0,crowd size=small,1782855050
2,PSE13151,2022-01-01,2022,1,Strategic developments,Strategic developments,Other,Military Forces of Israel (2021-2022),NaN,External/Other forces,Civilians (Palestine),Farmers (Palestine),Civilians,External/Other forces-Civilians,NaN,275,Middle East,Palestine,Gaza Strip,Khan Yunis,NaN,Al Qararah,31.3739,34.3409,2,Dunia Al Watan; Ma'an News Agency; Palestine N...,Other-National,"Other: On 1 January 2022, Israeli military for...",0,Foreign military engagement,1753954689
3,PSE13169,2022-01-01,2022,1,Political violence,Riots,Mob violence,Rioters (Israel),Settlers (Israel),Rioters,Civilians (Palestine),NaN,Civilians,Rioters-Civilians,Civilian targeting,275,Middle East,Palestine,West Bank,Hebron,NaN,Hebron,31.5294,35.0938,1,Ma'an News Agency; PCHR: Palestinian Centre fo...,Other-National,"On 1 January 2022, Israeli settlers hurled sto...",0,crowd size=small,1782855050
4,PSE23994,2022-01-01,2022,1,Strategic developments,Strategic developments,Other,Military Forces of Israel (2021-2022),NaN,External/Other forces,Civilians (Palestine),Farmers (Palestine),Civilians,External/Other forces-Civilians,NaN,275,Middle East,Palestine,Gaza Strip,Deir El Balah,NaN,Deir al Balah,31.4178,34.3503,2,PLO Negotiations Affairs Department,Other,"Other: On 1 January 2022, Israeli military for...",0,Foreign military engagement,1753954807


In [2]:
print('Dataset shape:', raw.shape)
raw.info()


Dataset shape: (58842, 31)
<class 'pandas.DataFrame'>
RangeIndex: 58842 entries, 0 to 58841
Data columns (total 31 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   event_id_cnty       58842 non-null  str    
 1   event_date          58842 non-null  str    
 2   year                58842 non-null  int64  
 3   time_precision      58842 non-null  int64  
 4   disorder_type       58842 non-null  str    
 5   event_type          58842 non-null  str    
 6   sub_event_type      58842 non-null  str    
 7   actor1              58842 non-null  str    
 8   assoc_actor_1       10444 non-null  str    
 9   inter1              58842 non-null  str    
 10  actor2              46483 non-null  str    
 11  assoc_actor_2       10902 non-null  str    
 12  inter2              46483 non-null  str    
 13  interaction         58842 non-null  str    
 14  civilian_targeting  19297 non-null  str    
 15  iso                 58842 non-null  i

In [3]:
missing_values = pd.DataFrame({
    'Missing count': raw.isnull().sum(),
    'Missing percentage': raw.isnull().mean() * 100
})
display(missing_values.round(2))

print('Exact duplicate rows:', raw.duplicated().sum())
print('Repeated event IDs:', raw.event_id_cnty.duplicated().sum())
print('Event-date range:', raw.event_date.min(), 'to', raw.event_date.max())


Exact duplicate rows: 0
Repeated event IDs: 0
Event-date range: 2022-01-01 to 2025-09-30


,Missing count,Missing percentage
event_id_cnty,0,0.00
event_date,0,0.00
year,0,0.00
time_precision,0,0.00
disorder_type,0,0.00
event_type,0,0.00
sub_event_type,0,0.00
actor1,0,0.00
assoc_actor_1,48398,82.25
inter1,0,0.00


### 1.4 Dataset Evaluation and Complexity

We selected ACLED event-level data because its actor labels, civilian-targeting flags, incident descriptions, and geographic/date precision support our research questions. Weekly aggregated data lack the incident detail we need. The UCDP-derived source-count dataset we considered addresses a different reporting-coverage problem and uses different event definitions and coverage. We chose ACLED as our principal dataset rather than combining records that could represent the same events.

Our dataset exceeds the guide's approximate minimums of 5,000 rows and 10 columns. It supports a practical decision about which locations and periods warrant further investigation. We consider the project feasible because we have defined a limited geographic and historical scope and selected two complementary technique families.

**Dataset Complexity Statement.** Our principal dataset contains 58,842 records and 31 fields, including narratives, dates, geospatial information, actor variants, and patterned missingness. Our initial inspection found that all 58,842 `admin3` values are absent, 12,359 second-actor fields are missing, 19,086 records have location precision codes above 1, and 308 have date precision codes above 1. We also identified 27 maritime records outside our land-based scope. We found no exact duplicate rows or repeated event IDs. These characteristics require us to interpret missingness using the coding rules, preserve original actor labels alongside any groupings, define geographic exclusions, and assess sensitivity to recording precision. We consider the dataset a candidate for the advanced-complexity tier because of these challenges; its final assessment will depend on how well we address them.


### 1.5 Data Dictionary

We summarized the imported data types and missingness in the initial inspection. In the dictionary below, we distinguish each field's analytical meaning from its imported type. We treat numeric identifiers and precision codes as labels rather than continuous measurements.

| Field | Analytical type | Meaning |
|---|---|---|
| `event_id_cnty` | Identifier | Event key. |
| `event_date` | Date | Coded event date. |
| `year` | Integer | Event year. |
| `time_precision` | Ordinal category | Date certainty code. |
| `disorder_type` | Category/multiple labels | Broad disorder grouping. |
| `event_type` | Category | Event class. |
| `sub_event_type` | Category | Detailed event class. |
| `actor1` | Category | First coded actor. |
| `assoc_actor_1` | Category/text | Actor-one associations. |
| `inter1` | Category | Actor-one type. |
| `actor2` | Category | Second coded actor. |
| `assoc_actor_2` | Category/text | Actor-two associations. |
| `inter2` | Category | Actor-two type. |
| `interaction` | Category | Actor-type pairing. |
| `civilian_targeting` | Flag/category | Recorded targeting label. |
| `iso` | Category/code | Territory code. |
| `region` | Category | Broad geographic area. |
| `country` | Category | Coded territory. |
| `admin1` | Category | First administrative level. |
| `admin2` | Category | Second administrative level. |
| `admin3` | Category | Third administrative level. |
| `location` | Category/text | Named event location. |
| `latitude` | Numeric | Coded latitude. |
| `longitude` | Numeric | Coded longitude. |
| `geo_precision` | Ordinal category | Location certainty code. |
| `source` | Text | Cited reporting sources. |
| `source_scale` | Category | Reporting scale labels. |
| `notes` | Text | Event narrative. |
| `fatalities` | Integer estimate | Reported event deaths. |
| `tags` | Category/text | Additional thematic labels. |
| `timestamp` | Integer/datetime | Record-update timestamp. |

Definitions: [ACLED codebook](https://acleddata.com/methodology/acled-codebook).

**Our interpretation of the coding:**

- The downloaded targeting label is `Civilian targeting`. A blank does not prove civilians were unharmed. Some civilians harmed during battles or attacks on armed groups are not represented by the targeting flag.
- `Violence against civilians` is one event type; the targeting field can identify relevant events in other types too.
- Actor order is not a universal attacker/victim ordering. For the *Violence against civilians* type, ACLED assigns the perpetrator and target by its coding rules. For other event types, we will review the coding rules and narrative before assigning actor roles.
- Coordinates identify coded places and can be approximate centroids, even at high precision. They are not verified street-level impact points.
- Fatalities are event totals, not automatically Palestinian civilian deaths or deaths caused by actor1. A recorded zero can reflect no fatality information rather than verified absence of deaths.


### 1.6 Scope and Initial Feasibility Findings

We defined a candidate study group using in-scope events with the `Civilian targeting` label and the recorded actor label `Civilians (Palestine)`. This group represents a coded subset of recorded events rather than all harm to Palestinian civilians. We will validate group membership and actor roles during preprocessing.

We identified 27 maritime records outside our land-based scope. We will exclude them from the analysis while retaining them in the original file. We will interpret absent actor/association fields and blank targeting labels using the codebook rather than guessing identities or assuming that civilians were unharmed. Because `admin3` is entirely absent, we will exclude it from the analysis instead of imputing locations.


In [4]:
inspection = raw.copy()
inspection['date_parsed'] = pd.to_datetime(raw.event_date, errors='coerce')
in_scope = raw.country.eq('Palestine') & raw.admin1.isin(['Gaza Strip', 'West Bank'])
scoped = inspection.loc[in_scope].copy()
target_flag = scoped.civilian_targeting.eq('Civilian targeting')
palestinian_civilian_label = (
    scoped.actor1.fillna('').eq('Civilians (Palestine)') |
    scoped.actor2.fillna('').eq('Civilians (Palestine)')
)
candidate = scoped.loc[target_flag & palestinian_civilian_label].copy()
display(pd.DataFrame({
    'in_scope_events': scoped.groupby('admin1').size(),
    'candidate_targeting_events': candidate.groupby('admin1').size(),
}))
print('Candidate cohort:', len(candidate))
print('Invalid dates:', int(inspection.date_parsed.isna().sum()))
print('Location precision codes:')
display(pd.crosstab(scoped.admin1, scoped.geo_precision))
candidate_periods = candidate.assign(period=candidate.date_parsed.ge('2023-10-07').map({False: 'Before Oct 7, 2023', True: 'Oct 7, 2023 onward'}))
display(pd.crosstab(candidate_periods.admin1, candidate_periods.period))
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == raw_hash, 'Original file was modified.'


Candidate cohort: 17047
Invalid dates: 0
Location precision codes:


,in_scope_events,candidate_targeting_events
admin1,,
Gaza Strip,26664,10714
West Bank,32151,6333


geo_precision,1,2,3
admin1,,,
Gaza Strip,14313,12231,120
West Bank,25440,6625,86


period,"Before Oct 7, 2023","Oct 7, 2023 onward"
admin1,,
Gaza Strip,63,10651
West Bank,1865,4468


Our candidate study group contains **17,047 records**: 10,714 in Gaza and 6,333 in the West Bank. Both regional subsets provide substantial event volume for our proposed analysis. However, we found only 63 candidate events in Gaza before October 7, 2023, compared with 10,651 afterward. We will account for this sparse earlier baseline when developing temporal thresholds. These initial counts establish feasibility; they do not demonstrate causation or establish a legal finding.


### 1.7 Preliminary Data Mining Plan

We will examine where recorded targeting of Palestinian civilians concentrates and when it becomes unusually frequent. We will analyse Gaza and the West Bank separately. Our initial variable choices are based on the research questions, field definitions, and observed data quality rather than feature-importance results.

**Data preparation.** We will convert `event_date` to a date, validate event IDs and coordinates, and exclude maritime records from the land-based analysis. We will define our study group using `civilian_targeting` and the Palestinian civilian actor labels. Any broader actor groupings will retain the original names so that our decisions remain traceable. We will preserve unknown actor information rather than infer identities.

**Candidate variables.** We selected the following fields for specific roles in our analysis:

| Variables | Our planned use |
|---|---|
| `latitude`, `longitude` | We will use these as the main inputs for geographic clustering. |
| `event_date` | We will group events into weeks, compare periods, and calculate previous-week and previous-four-week counts. |
| `admin1`, `admin2` | We will separate the two study areas and summarize activity by district and week. |
| `civilian_targeting`, `actor1`, `actor2` | We will use these to identify focal events and describe recorded actor involvement after reviewing actor roles. |
| `event_type`, `sub_event_type`, `disorder_type` | We will define the political-violence comparison group and describe the methods represented in clusters and unusual weeks. |
| `geo_precision`, `time_precision` | We will compare results with and without less precise records. These codes will serve as quality categories rather than numeric distances. |
| `fatalities` | We will consider this as a separate measure of reported event severity, recognizing that it does not automatically count Palestinian civilian deaths. |

We will retain `event_id_cnty` for record identification and duplicate checks, and use `notes` and `source` to review selected events. IDs, update timestamps, and narrative text will not be direct inputs to the two main models. We will exclude the entirely absent `admin3` field. We may consider additional fields if they address a specific research question.

**Analysis tables.** We will prepare an incident table with one row per focal event for geographic analysis. We will also prepare a district-week table containing focal-event counts, total recorded political-violence counts, and focal-event shares. We will calculate each share using the same district and week for its numerator and denominator. A zero denominator will result in an undefined share. We will review absent district-week combinations before treating them as zero recorded activity.

**Proposed methods.**

| Method | Main inputs | Our purpose | Comparison and evaluation |
|---|---|---|---|
| **DBSCAN spatial clustering** | Coordinates of focal events, separated by region and period. | We will identify geographic concentrations of recorded targeting. | We will compare radii of 2, 5, and 10 km and minimum cluster sizes of 5 or 10 events. We will examine cluster sizes, unclustered events, and consistency with district summaries and higher-precision subsets. |
| **Temporal anomaly detection** | District-week focal-event counts; targeting shares and previous-week/four-week counts for Isolation Forest. | We will identify unusually intense weeks and sustained increases. | We will compare a trailing median/MAD rule with Isolation Forest, vary historical windows and settings, and review selected alerts against incident records. |

**Scaling.** For DBSCAN, we will convert coordinates from degrees to radians and use haversine distance. We will not apply standard scaling to coordinates because we want distances to retain their geographic meaning. Isolation Forest and the median/MAD rule do not require standard scaling, so counts and shares can remain in their original units. If we later introduce a distance-based model with several numeric measures, we will fit its scaling on training data only. We will keep categorical labels separate from numeric distances.

**Validation.** We plan to use 2022 for initial fitting and January-September 2023 for selecting settings, followed by chronological evaluation on later periods. We will construct historical features from preceding weeks only. Any period already used for exploratory analysis, including the 2025 tree test, will be treated as observed validation data rather than an untouched final test set. Maps based on the full historical period will be presented as retrospective descriptions.

We will compare settings, regional results, and recording precision. Because Gaza's earlier targeting records are sparse, we will assess alternative historical windows and use a fallback count/quantile rule when the median/MAD baseline provides no useful threshold. We will distinguish zero recorded activity from evidence that no harm occurred.

**Expected outputs.** We will produce cluster maps, weekly timelines, method comparisons, and a traceable list of locations or periods for further investigation. Our conclusions will concern recorded patterns and their limitations rather than independently establishing causes, intent, or a legal finding of genocide.


### 1.8 References

1. ACLED. *Palestine event-level records, January 1, 2022-September 30, 2025*. Downloaded October 18, 2026 through the [Data Export Tool](https://acleddata.com/conflict-data/data-export-tool?grain=disaggregated). Source: ACLED (Armed Conflict Location & Event Data).
2. ACLED. [Codebook](https://acleddata.com/methodology/acled-codebook) and [Fatalities methodology](https://acleddata.com/methodology/fatalities). Definitions and interpretation limitations.
3. ACLED. [Export Tool guide](https://acleddata.com/use-access/data-export-tool-user-guide), [Terms and Conditions](https://acleddata.com/terms-and-conditions), and [Attribution Policy](https://acleddata.com/attributionpolicy).
4. DS312. *Final Project Guide*, A.Y. 2026-2027, pp. 1-3: Phase 1 requirements and dataset-selection criteria. Supplied course document.

Documentation consulted October 18, 2026. **Dataset/problem approval: awaiting instructor consultation.**
